# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

This section reviews two findings from the FlyRank research paper.

The purpose is not to grade the research or challenge the findings as incorrect.

Instead, I use the findings to practice asking the same methodology questions that should be asked of my own modeling work:

1. Where does the outcome or label come from?
2. Does the validation or study design support the strength of the claim?

The questions below are constructive and focus on what additional information would make the findings easier to interpret.

### Finding 1 — The CTR Cliff

The research describes a relationship between search position and click-through rate, including a sharp reduction in CTR as ranking position worsens.

### Methodology question

A useful methodology question is:

**How was the CTR outcome defined and aggregated across observations, and how was the analysis designed to separate ranking position from other factors that can affect CTR, such as query intent, search features, device mix, or differences between pages?**

This matters because an observed relationship between position and CTR does not by itself establish that position is the only reason CTR changes.

The finding is useful as directional evidence, but understanding the construction of the underlying observations and controls would help determine how broadly the result should be generalized.

### Finding 2 — Content Lifecycle / Growing vs Declining

The research analyzes content performance through lifecycle patterns, including differences between growing and declining content.

### Methodology question

A useful methodology question is:

**How was the growing-versus-declining label created, and what time window and threshold determined whether an observation was classified as growing or declining?**

This matters because changing the time window or threshold can change which pages receive each label.

A second useful validation question is whether the same content could appear in multiple lifecycle groups across different periods and how the study design handled repeated observations.

The finding can provide useful directional evidence about content lifecycle behavior, while the exact label construction and validation design determine how strongly the result should be interpreted.

## 2. My model under an honest split (before/after)

My Week-5 model uses March 2026 performance signals to predict an observed April 2026 decline outcome.

The original modeling comparison used a client-grouped split. In this audit, I compare that honest grouped design with an ordinary random row split.

The purpose is to demonstrate why validation design matters.

The random split can allow observations from the same client to appear in both training and test data.

The grouped split keeps each client entirely in either training or test data.

The grouped result is therefore the more appropriate estimate for this client-level decision-support question.

#### Imports

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

print("Libraries loaded.")

Libraries loaded.


In [2]:
# Reuse the same March/April files used in Week 5.
# If these variables already exist, this cell does not need to download them again.

from google.colab import userdata
from huggingface_hub import login, hf_hub_download

HF_TOKEN = userdata.get("HF_TOKEN")

try:
    login(token=HF_TOKEN)
except Exception:
    pass

march_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

april_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-04/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

print("March and April files ready.")

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  134MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

March and April files ready.


#### March features

In [3]:
feature_columns = [
    "report_date",
    "client_hash_id",
    "content_hash_id",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions",
    "sessions_organic",
    "sessions_ai",
    "scroll_events"
]

march_raw = pd.read_parquet(
    march_file,
    columns=feature_columns
)

march_features = (
    march_raw
    .groupby(
        ["client_hash_id", "content_hash_id"],
        as_index=False
    )
    .agg(
        march_impressions=("gsc_impressions", "sum"),
        march_clicks=("gsc_clicks", "sum"),
        march_avg_position=("gsc_avg_position", "mean"),
        march_sessions=("ga4_sessions", "sum"),
        march_engaged_sessions=("ga4_engaged_sessions", "sum"),
        march_organic_sessions=("sessions_organic", "sum"),
        march_ai_sessions=("sessions_ai", "sum"),
        march_scroll_events=("scroll_events", "sum")
    )
)

march_features["march_ctr"] = np.where(
    march_features["march_impressions"] > 0,
    march_features["march_clicks"] /
    march_features["march_impressions"],
    np.nan
)

march_features["march_engagement_rate"] = np.where(
    march_features["march_sessions"] > 0,
    march_features["march_engaged_sessions"] /
    march_features["march_sessions"],
    np.nan
)

print("Aggregated March rows:", len(march_features))

Aggregated March rows: 331437


#### April outcome

In [4]:
april_raw = pd.read_parquet(
    april_file,
    columns=[
        "report_date",
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "ga4_sessions",
        "ga4_engaged_sessions"
    ]
)

april_outcome = (
    april_raw
    .groupby(
        ["client_hash_id", "content_hash_id"],
        as_index=False
    )
    .agg(
        april_impressions=("gsc_impressions", "sum"),
        april_clicks=("gsc_clicks", "sum"),
        april_sessions=("ga4_sessions", "sum"),
        april_engaged_sessions=("ga4_engaged_sessions", "sum")
    )
)

model_df = march_features.merge(
    april_outcome,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

model_df["future_decline"] = np.where(
    model_df["march_clicks"] > 0,
    (model_df["april_clicks"] < model_df["march_clicks"]).astype(int),
    np.nan
)

model_df = model_df.dropna(
    subset=["future_decline"]
).copy()

model_df["future_decline"] = (
    model_df["future_decline"]
    .astype(int)
)

print("Model rows:", len(model_df))
print("Positive rate:", round(model_df["future_decline"].mean(), 4))

Model rows: 68837
Positive rate: 0.6552


### Define X, y, groups

This cell is important because it prevents the X is not defined error.

In [5]:
feature_cols = [
    "march_impressions",
    "march_clicks",
    "march_avg_position",
    "march_sessions",
    "march_engaged_sessions",
    "march_organic_sessions",
    "march_ai_sessions",
    "march_scroll_events",
    "march_ctr",
    "march_engagement_rate"
]

X = model_df[feature_cols].copy()
y = model_df["future_decline"].copy()
groups = model_df["client_hash_id"].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Unique clients:", groups.nunique())

X shape: (68837, 10)
y shape: (68837,)
Unique clients: 44


#### BEFORE — ordinary random row split




In [6]:
X_train_before, X_test_before, y_train_before, y_test_before = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

model_before = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        )
    ]
)

model_before.fit(
    X_train_before,
    y_train_before
)

prob_before = model_before.predict_proba(
    X_test_before
)[:, 1]

pred_before = (
    prob_before >= 0.50
).astype(int)

before_metrics = {
    "accuracy": accuracy_score(
        y_test_before,
        pred_before
    ),
    "precision": precision_score(
        y_test_before,
        pred_before,
        zero_division=0
    ),
    "recall": recall_score(
        y_test_before,
        pred_before,
        zero_division=0
    ),
    "f1": f1_score(
        y_test_before,
        pred_before,
        zero_division=0
    ),
    "roc_auc": roc_auc_score(
        y_test_before,
        prob_before
    )
}

print("BEFORE — random row split")
print(pd.Series(before_metrics).round(4))

BEFORE — random row split
accuracy     0.6550
precision    0.6555
recall       0.9978
f1           0.7912
roc_auc      0.5628
dtype: float64


### After: client-grouped split

The improved validation design keeps all observations from a client in only one partition.

This reduces the possibility that the model benefits from seeing other observations from the same client during training.

The grouped result is therefore treated as the more honest validation estimate for this modeling question.

In [7]:
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        X,
        y,
        groups=groups
    )
)

X_train_after = X.iloc[train_idx].copy()
X_test_after = X.iloc[test_idx].copy()

y_train_after = y.iloc[train_idx].copy()
y_test_after = y.iloc[test_idx].copy()

model_after = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        )
    ]
)

model_after.fit(
    X_train_after,
    y_train_after
)

prob_after = model_after.predict_proba(
    X_test_after
)[:, 1]

pred_after = (
    prob_after >= 0.50
).astype(int)

after_metrics = {
    "accuracy": accuracy_score(
        y_test_after,
        pred_after
    ),
    "precision": precision_score(
        y_test_after,
        pred_after,
        zero_division=0
    ),
    "recall": recall_score(
        y_test_after,
        pred_after,
        zero_division=0
    ),
    "f1": f1_score(
        y_test_after,
        pred_after,
        zero_division=0
    ),
    "roc_auc": roc_auc_score(
        y_test_after,
        prob_after
    )
}

print("AFTER — client-grouped split")
print(pd.Series(after_metrics).round(4))

print()
print("Train clients:", groups.iloc[train_idx].nunique())
print("Test clients:", groups.iloc[test_idx].nunique())

AFTER — client-grouped split
accuracy     0.6839
precision    0.6837
recall       0.9997
f1           0.8121
roc_auc      0.6132
dtype: float64

Train clients: 35
Test clients: 9


#### before/after table

In [8]:
before_after = pd.DataFrame([
    {
        "validation_design": "Random row split",
        **before_metrics
    },
    {
        "validation_design": "Client-grouped split",
        **after_metrics
    }
])

print("BEFORE vs AFTER")
display(before_after.round(4))

BEFORE vs AFTER


,validation_design,accuracy,precision,recall,f1,roc_auc
0,Random row split,0.6550,0.6555,0.9978,0.7912,0.5628
1,Client-grouped split,0.6839,0.6837,0.9997,0.8121,0.6132


#### verify no client overlap

In [9]:
train_clients = set(
    groups.iloc[train_idx]
)

test_clients = set(
    groups.iloc[test_idx]
)

client_overlap = train_clients.intersection(
    test_clients
)

print("Train/test client overlap:", len(client_overlap))

if len(client_overlap) == 0:
    print("Grouped validation check passed.")
else:
    print("WARNING: client overlap detected.")

Train/test client overlap: 0
Grouped validation check passed.


### Interpretation

The random-row result and the client-grouped result are intentionally shown together.

The difference between them demonstrates why validation design matters.

The grouped result is the more conservative estimate for this use case because the model is evaluated on clients that were not represented in training.

A change in performance between the two designs should be interpreted as evidence about validation sensitivity, not as proof that the model itself changed.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

The final model feature set is audited for information that would not have been available at the March decision point.

The audit checks:

- future April outcome fields;
- the future target;
- Week-4 product/action outputs;
- other label-like fields;
- client identifiers inside the numerical feature matrix.

A clean result means no obvious forbidden fields are included in the final feature list.

In [10]:
forbidden_features = [
    "april_impressions",
    "april_clicks",
    "april_sessions",
    "april_engaged_sessions",
    "future_decline",
    "baseline_action_score",
    "reason_code",
    "action",
    "refresh_tier",
    "health_score",
    "priority_score",
    "label",
    "target",
    "client_hash_id",
    "content_hash_id"
]

leakage_found = [
    column
    for column in feature_cols
    if column in forbidden_features
]

print("Final model features:")
print(feature_cols)

print()
print("Forbidden features found:")
print(leakage_found)

if not leakage_found:
    print("Feature leakage audit passed.")
else:
    print("WARNING: possible leakage detected.")

Final model features:
['march_impressions', 'march_clicks', 'march_avg_position', 'march_sessions', 'march_engaged_sessions', 'march_organic_sessions', 'march_ai_sessions', 'march_scroll_events', 'march_ctr', 'march_engagement_rate']

Forbidden features found:
[]
Feature leakage audit passed.


#### Target separation

In [11]:
print("Target column:", "future_decline")
print("Target included in X:", "future_decline" in X.columns)

if "future_decline" not in X.columns:
    print("Target separation check passed.")
else:
    print("WARNING: target appears in model features.")

Target column: future_decline
Target included in X: False
Target separation check passed.


#### Grouped leakage check

In [12]:
print("Train clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Client overlap:", len(client_overlap))

if len(client_overlap) == 0:
    print("Grouped validation leakage check passed.")

Train clients: 35
Test clients: 9
Client overlap: 0
Grouped validation leakage check passed.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Original Week-5 claim

"The Logistic Regression model provides directional decision-support for predicting future content decline and can help identify pages that should be refreshed."

### Why this claim is too strong

The model was evaluated on one March-to-April period and one client-grouped split.

The observed performance does not establish that the model will generalize to every future period or that using the model will cause content-refresh decisions to improve outcomes.

### Safer rewritten claim

"On the evaluated March-to-April dataset and client-grouped test split, the Logistic Regression model produced measured classification performance for the defined future-decline outcome. These results provide directional evidence that March performance signals can support a content-opportunity scoring workflow. The model should be treated as decision-support rather than proof that a page should be refreshed or that a refresh will improve performance."

### What remains uncertain

The evidence does not establish causal impact.

It also does not establish performance across different months, clients, datasets, or future operating conditions.

Additional time-based validation and repeated evaluation would be needed before making a stronger generalization.

## Self-check

- [x] I filled all five required sections.
- [x] I reviewed two findings from the FlyRank research paper.
- [x] I asked a methodology question about each finding.
- [x] I explained why label construction and validation design matter.
- [x] I compared an ordinary random split with a client-grouped split.
- [x] I used the same Week-5 modeling approach for the comparison.
- [x] I verified that train and test clients do not overlap.
- [x] I audited the final feature set for leakage.
- [x] I checked that the future target is not included in X.
- [x] I inspected the validation design rather than rewarding complexity.
- [x] I rewrote an overly strong claim using observed, measured, directional, and decision-support language.
- [x] I did not claim causation.
- [x] I did not claim universal generalization.
- [x] I did not include client names, URLs, private queries, or credentials.
- [x] The notebook is intended to run top to bottom without requiring variables from another notebook.
- [x] The notebook is saved as `work/notebooks/w06_validation_audit.ipynb`.